In [ ]:
import os
os.chdir(r"E:\MyAIProject") # workspace
from LinLanAIFrame import *
from LinLanAIFrame.load_pretrain_models import get_model_path

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
def init_model(model_name, download_path="./"):
    global device
    model_id, config_path, state_dict_path = get_model_path(model_name, download_path=download_path)
    model = init_model_from_config_state_dict(config_path=config_path, state_dict_path=state_dict_path, strict=False)
    for param in model.parameters():
        param.requires_grad = False
    model = model.to(device)
    model = model.eval()
    return model
vae_latent_std = 4.18157
clip = init_model("clip-224")
vae = init_model("vae-x8")
flow = init_model("dit")
print("Using device:", device)

In [ ]:
prompts = [
    # 生物与自然的错位
    ("A giraffe with a neck made of stacked vintage suitcases, walking through a foggy London street, cinematic lighting, photorealistic", "一只脖子由堆叠的古董手提箱组成的长颈鹿，漫步在雾蒙蒙的伦敦街头，电影布光，照片级真实"),
    ("A giant hummingbird with crystal wings pollinating glowing neon flowers on Mars, volcanic landscape, 8k, intricate details", "一只翅膀由水晶制成的大蜂鸟在火星上给发光的霓虹花朵授粉，火山地貌，8k，细节繁复"),
    ("An octopus playing a grand piano in a sunken ballroom, bubbles rising as musical notes, volumetric lighting, unreal engine 5", "一只章鱼在沉没的舞厅里弹奏大钢琴，气泡像音符一样升起，体积光，虚幻引擎5渲染"),
    ("A snail carrying a lighthouse on its back, leaving a trail of glowing stardust, night beach, magical realism, wide angle", "一只背着灯塔的蜗牛，身后留下发光的星尘轨迹，夜色海滩，魔幻现实主义，广角镜头"),
    ("A bonsai tree growing tiny hot air balloons instead of leaves, on a wooden table next to a cup of coffee, macro photography", "一棵盆景树没有长叶子，而是长出了微型热气球，放置在木桌上的咖啡杯旁，微距摄影"),
    ("A peacock with feathers made of stained glass, displaying in a dark cathedral, colored light projections, gothic surrealism", "一只羽毛由彩色玻璃制成的孔雀，在黑暗的大教堂中开屏，投射出斑斓的光影，哥特式超现实"),
    
    # 物体与场景的变异
    ("A grandfather clock melting into a river of honey, floating in a void of clouds, Salvador Dali style, surrealism", "一座落地钟融化成蜂蜜河流，漂浮在云层虚空中，萨尔瓦多·达利风格，超现实主义"),
    ("A skyscraper built entirely out of stacked vintage books, people jumping between floors like lily pads, whimsical architecture", "一座完全由堆叠的旧书建成的摩天大楼，人们在楼层间像跳荷叶一样跳跃，异想天开的建筑"),
    ("A vintage bicycle sailing through a sky of cotton candy clouds, basket full of glowing jellyfish, dreamy pastel colors", "一辆老式自行车航行在棉花糖云朵中，车篮里装满了发光的水母，梦幻的粉彩色调"),
    ("A typewriter where each key press releases a butterfly, sitting on a desk in a floating library, magical atmosphere, soft focus", "一台每按一个键就飞出一只蝴蝶的打字机，置于悬浮图书馆的书桌上，魔法氛围，柔焦"),
    ("A violin with strings made of spider silk, played by a robotic hand in a snow globe, delicate lighting, macro shot", "一把琴弦由蜘蛛丝制成的小提琴，在雪景玻璃球里被一只机械手演奏，精致布光，微距镜头"),
    ("A chandelier made of thousands of tiny glass eyes, hanging in a desert oasis at midnight, eerie glow, high definition", "一盏由数千只微小玻璃眼珠制成的水晶吊灯，悬挂在午夜的沙漠绿洲上，散发诡异光芒，高清画质"),
    
    # 经典形象的重塑
    ("A samurai warrior riding a mechanical T-Rex through a cyberpunk Tokyo alleyway, neon reflections, rain, cinematic composition", "一名武士骑着机械霸王龙穿过赛博朋克风格的东京小巷，霓虹倒影，雨水，电影构图"),
    ("A mermaid lounging on a crescent moon, combing her hair with a fork made of constellations, cosmic background, ethereal", "一名美人鱼懒洋洋地躺在新月上，用星座形状的梳子梳理头发，宇宙背景，空灵唯美"),
    ("A knight in shining armor riding a giant snail into battle against a horde of rabbits, medieval tapestry style, humorous", "一名身穿闪亮盔甲的骑士骑着巨型蜗牛，与一群兔子大军作战，中世纪挂毯风格，幽默诙谐"),
    ("A pirate captain steering a ship made of ice through a sea of liquid gold, treasure map tattoo glowing, fantasy art", "一名海盗船长驾驶着冰造的船只，航行在液态黄金的海洋上，身上的藏宝图纹身闪闪发光，奇幻艺术"),
    ("A geisha with a face painted like a starry night sky, holding an umbrella made of peacock feathers, cherry blossoms falling", "一名艺伎的脸涂成星夜天空的模样，手持孔雀羽毛制成的伞，樱花飘落"),
    ("A cowboy roping a tornado in the old west, dust storm, dramatic sunset lighting, hyper-realistic", "一名牛仔在美国旧西部套索住了一股龙卷风，沙尘暴，戏剧性的日落光线，超写实"),
    
    # 微观与宏观的混淆
    ("A flea circus under a magnifying glass, tiny acrobats jumping on a grain of rice, warm lamp light, miniature world", "放大镜下的跳蚤马戏团，微小的杂技演员在一粒米饭上跳跃，温暖的灯光，微观世界"),
    ("A mountain range shaped like a sleeping dragon, clouds forming its breath, aerial view, epic scale, matte painting", "形似沉睡巨龙的连绵山脉，云雾构成它的呼吸，航拍视角，史诗级规模，数字绘景"),
    ("A coral reef growing inside a glass terrarium on a desk, fish swimming in loops, bioluminescence, desktop ecosystem", "一片珊瑚礁生长在桌面的玻璃罩内，鱼儿游弋成圈，生物荧光，桌面生态系统"),
    ("A galaxy contained within a glass snowglobe being shaken by a giant hand, stars swirling, cosmic dust, macro cosmic", "一个玻璃雪花球里包含着整个星系，正被一只巨手摇晃，星辰旋转，宇宙尘埃，宏观宇宙"),
    ("A forest where the trees are giant mushrooms and the birds are tiny blimps, foggy morning, fantasy landscape", "一片森林里的树木全是巨型蘑菇，鸟儿全是微型飞艇，雾蒙蒙的清晨，奇幻风景"),
    ("A single dewdrop reflecting an entire cityscape, sitting on a blade of grass, tilt-shift photography, sharp focus", "一颗露珠映射出整座城市景观，悬垂在草叶上，移轴摄影，焦点锐利"),
    
    # 材质与物理的违背
    ("A solid stone statue weeping tears of liquid gold, cracking with vines growing out, dramatic chiaroscuro lighting", "一尊坚硬的石像流下液态黄金的眼泪，身上裂开缝隙长出藤蔓，强烈的明暗对比布光"),
    ("A glass elephant shattering into a thousand diamonds as it jumps into a lake, frozen motion, high speed photography", "一只玻璃大象跳入湖面时碎裂成无数钻石，瞬间定格，高速摄影"),
    ("A clockwork heart beating inside a birdcage, gears turning, smoke coming out, steampunk aesthetic, brown tones", "一只发条心脏在鸟笼里跳动，齿轮转动，冒出烟雾，蒸汽朋克美学，棕色调"),
    ("A waterfall flowing upwards into the clouds, rainbow connecting the streams, gravity defying, fantasy concept art", "一道瀑布向上流入云端，水流间架起彩虹，反重力，奇幻概念艺术"),
    ("A house made of gingerbread with windows of melted sugar, chocolate syrup rain, sweet texture, close up", "一座姜饼做成的房子，窗户是融化的糖块，下着巧克力酱雨，甜美的质感，特写镜头"),
    ("A balloon animal dog chasing a solid metal bone across a field of fluffy clouds, weightless, playful, bright colors", "一只气球扭成的动物狗在棉花糖般的云野上追逐一根实心金属骨头，失重感，顽皮，色彩明亮"),
    
    # 纯视觉奇观
    ("A staircase spiraling infinitely into a void of geometric shapes, Escher style, black and white, optical illusion", "一座楼梯无限螺旋进入几何形状的虚空，埃舍尔风格，黑白，视觉错觉"),
    ("A whale swimming through a sea of clouds above a city skyline, golden hour lighting, majestic, wide panoramic shot", "一头鲸鱼在城市天际线的云海中游弋，金色时刻的光线，雄伟壮观，宽幅全景"),
    ("A garden where flowers bloom into tiny galaxies, soil made of crushed gemstones, ultraviolet photography, vibrant", "一座花园里的花朵绽放成微型星系，土壤由碎宝石组成，紫外线摄影，色彩鲜艳"),
    ("A mirror maze reflecting endless versions of a burning candle, flickering light, claustrophobic, mysterious", "一座镜面迷宫反射出无数燃烧的蜡烛影像，闪烁的光芒，幽闭恐惧，神秘莫测"),
    ("A bridge made of piano keys spanning a river of music notes, people walking creating melodies, abstract, artistic", "一座由钢琴键构成的桥梁横跨音符之河，行人走过奏响旋律，抽象，艺术感"),
    ("A swarm of fireflies creating a moving constellation in the shape of a phoenix, dark forest background, magical glow", "一群萤火虫组成了一只凤凰形状的流动星座，黑暗森林背景，魔法般的辉光"),
]
length = len(prompts)

In [ ]:
bpe=BPE(load=True)
def text_embed(caption:str, batch_size:int=4):
    token = bpe.encode_sentences([caption], dim=200, numpy=True)
    token = torch.from_numpy(token).expand(batch_size, 200).to("cuda")
    with torch.no_grad(), autocast():
        token_pool, token_global = clip.text_encoder.encode_text(token)
    return token_pool, token_global
def generator(batch_size, caption, mode="euler", step_nums=100):
    with torch.inference_mode(), autocast():
        token_pool, token_global = text_embed(caption, batch_size)
        latent = flow.sample(batch_size=batch_size, condition1=token_global, condition2=token_pool, mode=mode, step_nums=step_nums, image_shape=(16,64,64))
        latent = latent * vae_latent_std
        out = vae.latent2image(latent)
        save_image(out.detach().cpu().clamp(min=-1, max=1), nrow=int(math.sqrt(batch_size)), normalize=True, padding=1, fp=f"./generator.jpg")
        out = make_grid(out.detach().cpu().clamp(min=-1, max=1), nrow=int(math.sqrt(batch_size)), normalize=True, padding=1).permute(1,2,0).numpy()
        return out

In [ ]:
length = len(prompts)
batch_size = 9
mode = "euler"
index = random.randint(0, length-1)
caption = prompts[index][0]
print("中文:", prompts[index][1])
# caption = "A red Ferrari sports car parked in a city parking lot at night, with lit buildings in the background"
print("英文:", caption)
image = generator(batch_size=batch_size, caption=caption, mode=mode, step_nums=100).astype(np.float32)
plt.imshow(image)
plt.axis("off")
plt.show()

In [ ]:
batch_size = 9
mode = "euler"
caption = "A red Ferrari sports car parked in a city parking lot at night, with lit buildings in the background"
print("英文:", caption)
image = generator(batch_size=batch_size, caption=caption, mode=mode, step_nums=100).astype(np.float32)
plt.imshow(image)
plt.axis("off")
plt.show()

In [ ]:
quit()